# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore, process, and analyze the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is structured according to the [Croissant](https://mlcommons.org/croissant) specification. Its Croissant schema can be found at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant schema (JSON-LD)
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display main metadata fields
md = dataset.metadata  # metadata is a single object
print(f"Dataset title: {md.name}")
print(f"Description: {md.description}\n")
print(f"Authors: {[author for author in getattr(md, 'author', [])]}")
print(f"Published: {getattr(md, 'datePublished', 'Unknown')}")
print(f"License: {md.license}")

## 2. Data Overview

Let's inspect the record sets defined in this dataset. Each record set, field, and column should be referenced by their Croissant `@id`.

**Note:** If available, `mlcroissant` provides the list of record sets and their IDs via `dataset.record_sets`.

In [ ]:
# List all record sets by their @id and human-friendly name (if available)
from pprint import pprint

if hasattr(dataset, "record_sets"):
    print("Available record sets (by @id):")
    for rs in dataset.record_sets:
        rs_id = rs['@id']
        rs_name = rs.get('name', '(no name)')
        print(f"- {rs_id} | name: {rs_name}")
else:
    print("No record sets found via dataset.record_sets (this may be an old-style Croissant schema).")

# For demonstration, list the fields for the FIRST record set, if available
record_sets = getattr(dataset, 'record_sets', [])
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nSample of records for record set: {first_record_set_id}")
    recs = list(dataset.records(record_set=first_record_set_id))
    pprint(recs[0] if recs else "No records available.")
else:
    print("No record sets found in dataset metadata.")

## 3. Data Extraction

Load tables from all available record sets into Pandas DataFrames.

**Note:** All Croissant entities are always referenced by their `@id`.

In [ ]:
# Get the list of record set @ids for extraction
record_sets = getattr(dataset, 'record_sets', [])
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

# Load each record set into a pandas DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")

if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No available record sets to extract.")

## 4. Exploratory Data Analysis (EDA)

Let's try filtering and transforming some clinical data. We'll reference fields by their `@id` as required.

In [ ]:
# Choose the main record set by @id
main_record_set_id = record_set_ids[0] if record_set_ids else None
if not main_record_set_id:
    print("No main record set available.")
else:
    df = dataframes[main_record_set_id]
    print(f"DataFrame shape: {df.shape}")
    print(f"Sample columns: {df.columns.tolist()}")

    # Try to guess a numeric field '@id' (commonly columns with int/float)
    numeric_columns = df.select_dtypes(include=['number']).columns
    if len(numeric_columns) == 0:
        print("No numeric field detected for EDA.")
    else:
        numeric_field = numeric_columns[0]  # Use the first numeric column
        print(f"Using numeric field (by @id): {numeric_field}")

        # Set a threshold for demonstration
        threshold = df[numeric_field].quantile(0.5)  # use median as threshold

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical field (string/object dtype)
        group_cols = df.select_dtypes(include=['object']).columns
        group_field = None
        for col in group_cols:
            if col != numeric_field and df[col].nunique() < df.shape[0] / 2:
                group_field = col
                break
        if group_field:
            print(f"Grouping by field (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            display(grouped_df.head())
        else:
            print("No suitable group field found.")

## 5. Visualization

Visualize the distribution of the numeric field and show its values grouped by a key clinical attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_record_set_id or len(numeric_columns) == 0:
    print("No data available for visualization.")
else:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group_field (if available)
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=35, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we explored the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors dataset using the `mlcroissant` library. By referencing all entities by their Croissant `@id`, we loaded clinical records, examined key variables, and visualized major quantitative trends. The approach demonstrated here can be adapted to other Croissant-compliant datasets for rapid FAIR data exploration and analysis.